## LOAD PDF FILES

In [10]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

In [13]:
### PyPDFLoader
print("PyPDFLoader")
try:
    PyPDF_Loader = PyPDFLoader("data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf")
    pypdf_docs= PyPDF_Loader.load()
    print(pypdf_docs)
    print(f"Loaded {len(pypdf_docs)} pages")
    print(f"Page 1 content:{pypdf_docs[0].page_content[:100]}...")
    print(f"Metadata: {pypdf_docs[0].metadata}")
    
except Exception as e:
    print(f"Error : {e}")
    

PyPDFLoader
[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 17 for Word', 'creationdate': '2021-06-16T13:08:36-04:00', 'author': 'FDA/CBER', 'company': '', 'contenttypeid': '0x010100076A08FBD5310342BF958ADCB470BE9A', 'created': 'D:20171219', 'keywords': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry, CBER, CDER, Biologics, Drugs', 'lastsaved': 'D:20180507', 'moddate': '2021-06-16T16:17:42-04:00', 'sourcemodified': 'D:20210616132127', 'subject': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry', 'title': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry', '_dlc_dociditemguid': '3cf4e246-74a9-4f1f-ab34-2f63f0cab5d1', 'source': 'data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf', 'total_

In [16]:
# Method 2 PyMuPDFLoader (Fast and Accurate)
print("\n PyMuPDFLoader")
try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf")
    pymupdf_docs = pymupdf_loader.load()
    print(f"Loaded {len(pymupdf_docs)} pages")
    print(f"Included detailed metadata")
    print(pymupdf_docs)
    
except Exception as e:
    print(f"Error: {e}")


 PyMuPDFLoader
Loaded 49 pages
Included detailed metadata
[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 17 for Word', 'creationdate': '2021-06-16T13:08:36-04:00', 'source': 'data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf', 'file_path': 'data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf', 'total_pages': 49, 'format': 'PDF 1.5', 'title': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry', 'author': 'FDA/CBER', 'subject': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry', 'keywords': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry, CBER, CDER, Biologics, Drugs', 'moddate': '2021-06-16T16:17:42-04:00', 'trapped': '', 'modDate': "D:20210616161742-04'00

#### PDF Loader comparision

### PyPDFLoader:
- Simple and reliable ✅
- Good for most PDFs ✅
- Preservers Page numbers ✅
- Basic text extraction ❎
- use when: Standard text PDFs

### PyMuPDFLoader:
- Fast Processing ✅
- Good text extraction ✅
- Image extraction Support ✅
- Use when speed is important


### Handling PDF challenges

##### Purpose of this section PDFs are notoriously difficult to parse because they:
- Store Text in Complex ways(not just simple text)
- Can have formatting issues
- May contain scanned images(requiring OCR)
- Often have extraction artifacts

In [17]:
raw_text_pdf = """


    The financial performance fro fiscal year 2024
    shows significant growth in profitability.
    
    
    
    revenue increased by 25%.
    
The company's efficiency improved due to workflow
optimizATION.



Page 1 to 10
"""

def clean_text(text):
    text = " ".join(text.split())
    text = text.replace("Fi", "fi")
    text = text.replace("fi","f1")
    return text

cleaned = clean_text(raw_text_pdf)
print("BEFORE:")
print(repr(raw_text_pdf[:100]))
print(f"\nAFTER:")
print(repr(cleaned[:100]))

BEFORE:
'\n\n\n    The financial performance fro fiscal year 2024\n    shows significant growth in profitability.'

AFTER:
'The f1nancial performance fro f1scal year 2024 shows signif1cant growth in prof1tability. revenue in'


In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [19]:
from typing import List, Dict, Any
from langchain_core.documents import Document

class SmartPDFProcessor:
    """Advanced PDF Processing with error handling"""
    def __init__(self, chunk_size=1000,chunk_overlap=100):
        self.chunk_size = chunk_size,
        self.chunk_overlap = chunk_overlap,
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators=[" "]
        )
        
    def process_pdf(self, pdf_path:str)->List[Document]: 
        """Process PDF with smart chunking and metadta"""
        ## Load pdf
        loader= PyPDFLoader(pdf_path)
        pages=loader.load()
        processed_chunks=[]
        for page_num, page in enumerate(pages):
            #clean text
            cleaned_text=self._clean_text(page.page_content)
            
            #skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue
            
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[
                    {
                        **page.metadata,
                        "page": page_num +1,
                        "total_pages" : len(pages),
                        "chunk_method": "smart_pdf_processor",
                        "char_count": len(cleaned_text)
                    }
                ]    
            )
            processed_chunks.extend(chunks)
            
        return processed_chunks
    
    
    
    def _clean_text(self, text:str)-> str:
        text = " ".join(text.split())
        text = text.replace("Fi", "fi")
        text = text.replace("fi","f1")
        return text
        
        

In [20]:
preprocessor =  SmartPDFProcessor()
preprocessor


In [21]:
## Process a pdf if available
try:
    smart_chunks= preprocessor.process_pdf("data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")
    
except Exception as e:
    print(f"Error {e}")

Processed into 113 smart chunks


In [23]:
smart_chunks[0].metadata

{'producer': 'Adobe PDF Library 15.0',
 'creator': 'Acrobat PDFMaker 17 for Word',
 'creationdate': '2021-06-16T13:08:36-04:00',
 'author': 'FDA/CBER',
 'company': '',
 'contenttypeid': '0x010100076A08FBD5310342BF958ADCB470BE9A',
 'created': 'D:20171219',
 'keywords': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry, CBER, CDER, Biologics, Drugs',
 'lastsaved': 'D:20180507',
 'moddate': '2021-06-16T16:17:42-04:00',
 'sourcemodified': 'D:20210616132127',
 'subject': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry',
 'title': 'Chemistry, Manufacturing, and Controls Changes to an Approved Application: Certain Biological Products; Guidance for Industry',
 '_dlc_dociditemguid': '3cf4e246-74a9-4f1f-ab34-2f63f0cab5d1',
 'source': 'data/pdf/Chemistry_Manufacturing_Controls_Changes_Application_Final_06-16-2021.pdf',
 'total_pages': 49,
 'pa